## CS310 Natural Language Processing
## Lab 7: Decoding with LLMs

In this lab, we will practice some decoding strategies for LLM inference.

Download the model weights here: https://huggingface.co/Qwen/Qwen3-0.6B. Or you can load the model with path "Qwen/Qwen3-0.6B" in `transformers`.

In [1]:
import torch
import torch.nn.functional as F
import random
import numpy as np
import time

## T1. Explore the Model

First, we will explore the Qwen3-0.6B model and get familiar with some of the APIs of `transformers`.

In [1]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache

CANDIDATE_MODEL_PATHS = [
    "E:/CS310-Natural-Language-Processing/lab/lab7/Qwen3-0.6B",
    "lab/lab7/Qwen3-0.6B",
    ]

MODEL_PATH = "Qwen/Qwen3-0.6B"
for p in CANDIDATE_MODEL_PATHS:
    if p.startswith("Qwen/") or Path(p).exists():
        MODEL_PATH = p
        break

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# Evaluation mode
model = model.eval()

print('Loaded model path:', MODEL_PATH)
print('vocab size:', tokenizer.vocab_size)
print(f'special token {tokenizer.pad_token}:', tokenizer.pad_token_id)
print(f'special token id {tokenizer.pad_token_id}:', tokenizer.pad_token_id)

e:\CS310-Natural-Language-Processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 2439.22it/s]


Loaded model path: E:/CS310-Natural-Language-Processing/lab/lab7/Qwen3-0.6B
vocab size: 151643
special token <|endoftext|>: 151643
special token id 151643: 151643


### Note on output differences

If your greedy output is different from the instructor's expected text, the code is not necessarily wrong. The usual causes are:

- the local model weights are not exactly the same checkpoint as the instructor's;
- the tokenizer files or `transformers` version differ;
- notebook cells were executed out of order, so the kernel state is not identical;
- the example output in the lab is a reference run, not a strict cross-environment guarantee.

For greedy decoding, the result is deterministic only when the model, tokenizer, code, and runtime environment all match exactly.

The tokenizer can return the token IDs and the attention mask that indicates which tokens are padding tokens (`1` for real tokens, `0` for padding tokens).

Since we only have one sentence in the "batch", there is no padding used, and thus no `0` in the attention mask.

In [2]:
input_text = '学而时习之，不亦说乎！'
input_encoded = tokenizer(input_text, return_tensors="pt")

print('input ids:', input_encoded['input_ids'])
print('input attention mask:', input_encoded['attention_mask'])

# Map token ids back to tokens
input_ids = input_encoded['input_ids'][0]

input_tokens = tokenizer.convert_ids_to_tokens(input_encoded['input_ids'][0])
print('input tokens:', input_tokens)

input_token_strings = [tokenizer.convert_tokens_to_string([tok]) for tok in input_tokens]
print('input token strings:', input_token_strings)

input ids: tensor([[ 47764,  68536,  13343,  99347,  53930,   3837,  16530, 103972,  36587,
          99587,   6313]])
input attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
input tokens: ['åŃ¦', 'èĢĮ', 'æĹ¶', 'ä¹ł', 'ä¹ĭ', 'ï¼Į', 'ä¸į', 'äº¦', 'è¯´', 'ä¹İ', 'ï¼ģ']
input token strings: ['学', '而', '时', '习', '之', '，', '不', '亦', '说', '乎', '！']


It's easy to directly use the `generate` method to generate some sentences.

The `generate` method takes multiple arguments as input. As a Python syntactic sugar, you can use the `**` operator to unpack a dictionary and pass the key-value pairs as individual keyword arguments.

For instance, `model.generate(**input_encoded)` is equivalent to `model.generate(input_ids=input_encoded['input_ids'], attention_mask=input_encoded['attention_mask'])`, in which `input_encoded` is a dictionary.

Same applies to the forward function of the model.

In [3]:
input_text = "子曰：人"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
n_outputs = 5

output = model.generate(**input_encoded, 
                    max_length=30, 
                    num_return_sequences=n_outputs,
                    do_sample=True, 
                    top_k=50, 
                    top_p=0.95, 
                    temperature=0.7,
                    pad_token_id=0,
                )
# print(type(output))
# print(output.shape)

for i in range(n_outputs):
    output_text = tokenizer.decode(output[i], skip_special_tokens=True)
    output_text = output_text.replace("\n", "")
    print(output_text)

子曰：人之为事，广博而深，人之才有所不逮，故而有大者。子曰：虽
子曰：人之重于己，有三：有志于高，有行于上，有德于义。夫子之道
子曰：人之为贵，有一言，即己所为。子曰：人之为贵，有言，即己所
子曰：人之为贵，贵其所成。人之为贵，贵其所成。人之为贵，贵其所成。
子曰：人而无信，不知其可也。有信，则思有信。孔子的这句话表达的是一种怎样的态度？


We can see that the generation is far from perfect. It still has good chances to produce a lot of repetitions.

---

## T1. Implement Greedy Decoding 

Here we will practice implementing greedy decoding (top-1 sampling) manually.

Let's first pass the `input_encoded` dictionary to the `forward` function, and get the returned logits.

In [4]:
input_text = "今天天气"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)

output = model(**input_encoded)
# The above code is equivalent to:
# output = model(input_ids=input_encoded["input_ids"], 
#               attention_mask=input_encoded["attention_mask"])

print("output.keys():", output.keys())
logits = output.logits
print(logits.shape)

output.keys(): odict_keys(['logits', 'past_key_values'])
torch.Size([1, 2, 151936])


The returned `logits` tensor is of shape `(B, L, V)`, which contains the logits of all tokens. 

- `B` is batch size (here B=1);
- `L` is the length of the sequence (here L=2);
- `V` is the vocabulary size (here V=151936).

In order to conduct greedy decoding, i.e., top-1 sampling, we need to get the logits of the last token, by specifying the second dimension.

The size of the returned logits should be of the vocabulary size.

In [5]:
### START YOUR CODE ###
# Get the probability distribution predicted at the last token's position
last_token_logits = logits[0, -1, :]
### END YOUR CODE ###

# Test
print("last_token_logits.shape:", last_token_logits.shape)

# You expect to see the following output:
# last_token_logits.shape: torch.Size([151936])

last_token_logits.shape: torch.Size([151936])


Then, you simply pick the most likely token id from this vocabulary-sized distribution, by using the `argmax()` function.

In [8]:
### START YOUR CODE ###
import torch
top1_next_id = torch.argmax(last_token_logits, dim=-1)
### END YOUR CODE ###

# Test
print("top-1 next token id:", top1_next_id.item())

top1_next_str = tokenizer.decode(top1_next_id)
print("top-1 next token:", top1_next_str)

# You should expect to see the following output:
# top-1 next token id: 105212
# top-1 next token: 晴

top-1 next token id: 105212
top-1 next token: 晴


Next, you can all the model's `forward()` function to generate an output, with KV cache manually enabled.

- `past_key_values` is an instance of `DynamicCache`, which should be passed as an argument to the `forward()`.
- specify the argument `use_cache=True`.

In [9]:
input_text = "今天天气"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
input_ids = input_encoded["input_ids"]
attn_mask = input_encoded["attention_mask"]

# initialize KV cache
past_key_values = DynamicCache()
print("Before forward pass\nKV cache size in # of layers:", len(past_key_values.layers))

# forward pass 
### START YOUR CODE ###
output = model(
    input_ids=input_ids,
    attention_mask=attn_mask,
    past_key_values=past_key_values,
    use_cache=True,
 )
### END YOUR CODE ###
print()

# after forward pass, KV cache is not empty
print("After forward pass\nKV cache size in # of layers", len(past_key_values.layers))
if len(past_key_values.layers) > 0:
    print("shape of each layer:", past_key_values.layers[0].values.shape) # KV cache is of shape [batch_size, num_heads, seq_len, head_dim]

Before forward pass
KV cache size in # of layers: 0

After forward pass
KV cache size in # of layers 28
shape of each layer: torch.Size([1, 8, 2, 128])


Next, you can now implement the full generation loop with KV cache enabled, in which there is a generation loop.

- The loop continues until `max_gen_len` is reached, or a `"<endoftext>"` token is generated.
- For each loop iteration, use greedy decoding to sample a `next_id` from the output, and append it to `generated_ids`.
- Update `attention_mask` by increment it with a constant `[1]` tensor (with the correct dimension).
- Update `input_encoded` for each iteration, with `next_id` as the new `input_ids`, and the updated `attention_mask`.

In [10]:
def generate_greedy_with_kv_cache(input_text, model, tokenizer, max_gen_len=100):
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded["input_ids"]

    # initialize KV cache
    past_key_values = DynamicCache()
    
    # to store generated token ids
    generated_ids = input_ids

    # loop until max_gen_len is reached, or a `"<endoftext>"` token is generated
    count = 0
    while count < max_gen_len:
        ### START YOUR CODE ###
        output = model(
            input_ids=input_encoded["input_ids"],
            attention_mask=input_encoded["attention_mask"],
            past_key_values=past_key_values,
            use_cache=True,
        )

        # Greedy decoding
        next_logits = output.logits[0, -1, :]
        next_id = torch.argmax(next_logits, dim=-1).view(1, 1)
        generated_ids = torch.cat([generated_ids, next_id], dim=1) # append next_id to generated_ids

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"] 
        attention_mask = torch.cat([
            attention_mask,
            torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device),
        ], dim=1) # update attention_mask
        input_encoded = {"input_ids": next_id, "attention_mask": attention_mask} # update input_encoded

        # break if '<|endoftext|>' is generated
        if next_id.item() == tokenizer.pad_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

In [57]:
# Test
import time
SPECIAL_TOKEN_IDS = set([ tokenizer.pad_token_id])

input_text = "今天天气不错哟"

start_time = time.time()
generated_ids = generate_greedy_with_kv_cache(input_text, model, tokenizer, max_gen_len=100)
end_time = time.time()
print("generation speed: {:.4} tokens/s".format(generated_ids.shape[1] / (end_time - start_time)))

# Decode the generated tokens ids
print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

# You should expect to see the following output:
# generation speed: xx.xx tokens/s
# input text: 今天天气
# generated text:
# 今天天气不错哟，我们去公园散步吧。这句话中，"我们"指的是谁呢？A. 你 B. 我 C. 你和我 D. 你和我以及我
# 答案：C
# 问题：下面哪个选项是正确的？A. 你 B. 我 C. 你和我 D. 你和我以及我
# 答案：C
# 问题：下面哪个选项是正确的？A. 你 B. 我 C. 你和

generation speed: 6.181 tokens/s
input text: 今天天气不错哟
generated text:
今天天气不错哟，我们去公园散步吧。这句话中，"我们"指的是谁呢？A. 一个孩子 B. 一个孩子和一个成年人 C. 一个孩子 D. 一个成年人
答案：D
答案：D
答案：D
答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D


Let's compare it with the `generate()` method provided by `transformers` library.

In [56]:
# Test
input_text = "今天天气不错哟"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)

start_time = time.time()
generated_ids = model.generate(**input_encoded, 
                              max_length=100, 
                              num_return_sequences=1,
                              do_sample=False,
                              use_cache=True
                            )
end_time = time.time()
print("generation speed: {:.4} tokens/s".format(generated_ids.shape[1] / (end_time - start_time)))

# Decode the generated tokens ids
print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

generation speed: 7.602 tokens/s
input text: 今天天气不错哟
generated text:
今天天气不错哟，我们去公园散步吧。这句话中，"我们"指的是谁呢？A. 一个孩子 B. 一个孩子和一个成年人 C. 一个孩子 D. 一个成年人
答案：D
答案：D
答案：D
答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D

答案：D


## T2. Implement Top-k Sampling

Now, let's implement a `top-k` sampling algorithm. We will first practice the naive idea of **uniformly** sampling from top-k most likely next tokens. 

You need to use the results returned from the PyTorch tensor built-in `topk()` method to get the top-k logits and the corresponding indices. 

In the following example, you can check the top 5 most likely words following the sentence "今天天气":

In [16]:
k = 5
input_text = "今天天气"
input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)

output = model(**input_encoded)
logits = output.logits

### START YOUR CODE ###
last_token_logits = logits[0, -1, :]
topk_logits, topk_indices = torch.topk(last_token_logits, k=k)
### END YOUR CODE ###

# Test
print(topk_logits)
print(topk_indices)

for i in range(k):
    tok_id = topk_indices[i].item()
    tok = tokenizer.convert_ids_to_tokens(tok_id)
    tok_str = tokenizer.convert_tokens_to_string([tok])
    print(tok_str, end=' ')

# You should expect to see the following output:
# tensor([14.5000, 14.1875, 13.8125, 13.6250, 13.1875], dtype=torch.bfloat16,
#        grad_fn=<TopkBackward0>)
# tensor([105212, 115744,  20412, 104472,  42140])
# 晴 预报 是 怎么样 多 

tensor([14.3750, 14.1250, 13.7500, 13.5625, 13.1875], dtype=torch.bfloat16,
       grad_fn=<TopkBackward0>)
tensor([105212, 115744,  20412, 104472,  42140])
晴 预报 是 怎么样 多 

Let's integrate the top-k sampling into the generation process. 

The uniform sampling can be implemented using `random.choices` among the top-k indices.

In [35]:
def generate_topk_uniform(input_text, model, tokenizer, k=5, max_gen_len=50):
    '''
    Generate tokens from the top-k logits, and yield the sampled token id.
    Tokens are sampled from a naive uniform distribution.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded["input_ids"]

    # initialize KV cache
    past_key_values = DynamicCache()
    
    # to store generated token ids
    generated_ids = input_ids

    # loop until max_gen_len is reached, or a `"<endoftext>"` token is generated
    count = 0
    while count < max_gen_len:
        ### START YOUR CODE ###
        import torch
        import random

        output = model(
            input_ids=input_encoded["input_ids"],
            attention_mask=input_encoded["attention_mask"],
            past_key_values=past_key_values,
            use_cache=True,
        )

        # Top-k sampling (with naive uniform distribution)
        next_logits = output.logits[0, -1, :]
        topk_logits, topk_indices = torch.topk(next_logits, k=min(k, next_logits.size(-1))) # use topk()
        sampled_id = random.choices(topk_indices.tolist(), k=1)[0]
        next_id = torch.tensor([[sampled_id]], dtype=input_ids.dtype, device=input_ids.device) # use random.choices()
        generated_ids = torch.cat([generated_ids, next_id], dim=1)

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = torch.cat([
            attention_mask,
            torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device),
        ], dim=1)
        input_encoded = {"input_ids": next_id, "attention_mask": attention_mask}

        # break if '<|endoftext|>' is generated
        if next_id.item() == tokenizer.pad_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

In [36]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_uniform(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟去这边下大马路哦？ 衖小弟啊哦！ 栕名后面的一元加上 噶然克吗嘎还是后面一文格

是这个了吗 栖宅居小童

原词原格韵：

歌


In [37]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_uniform(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人学假舌道矣然且也己可欲而初，求仁义无由易乐其形悦豫为之谓？吾乐天下他人心念必为之乃乐乃益耶夫天至情死以己为人则


We can note that although the above uniform top-k sampling solves repetition issue, it will however produce *extremely incoherent* text. 

We should fix this by using a **proportional sampling** instead of uniform sampling.

There are plenty of different ways to implement proportionaly sampling. You can either:
- Create list of cumulative relative probabilities of the top k tokens. For instance, if the relative probabilities of $k=5$ tokens are $0.1$, $0.2$, $0.5$, $0.1$, and $0.1$, then you cumulative probability list is `cum_prob = [0.1, 0.3, 0.8, 0.9, 1.0]`. 
- Then you draw a random number $r$ from the unifrom distribution $[0,1]$ by `random.random()`, and you decide which token is sampled by telling which bin of `cum_prob` that $r$ falls into.
- Or instead, you use the `torch.multinomial()` function to accomplish similar sampling. *Note* the input weight provided to `torch.multinomial` should be the relative probabilities of the top $k$ tokens, which can be obtained from applying softmax to the logits. 

In [38]:
def generate_topk_proportion(input_text, model, tokenizer, k=50, max_gen_len=50):
    '''
    Generate tokens from the top-k logits, and yield the sampled token id.
    Tokens are sampled proportional to their logits.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded["input_ids"]

    # initialize KV cache
    past_key_values = DynamicCache()
    
    # to store generated token ids
    generated_ids = input_ids

    # loop until max_gen_len is reached, or a `"<endoftext>"` token is generated
    count = 0
    while count < max_gen_len:
        ### START YOUR CODE ###
        import torch
        import torch.nn.functional as F

        output = model(
            input_ids=input_encoded["input_ids"],
            attention_mask=input_encoded["attention_mask"],
            past_key_values=past_key_values,
            use_cache=True,
        )

        # Top-k sampling (with proportional sampling)
        last_token_logits = output.logits[0, -1, :]
        topk_logits, topk_indices = torch.topk(last_token_logits, k=min(k, last_token_logits.size(-1)))
        topk_probs = F.softmax(topk_logits, dim=-1) # use softmax

        sampled_pos = torch.multinomial(topk_probs, num_samples=1)
        next_id = topk_indices[sampled_pos].view(1, 1) # Use the manual way of propotional sampling, or use torch.multinomial()
        generated_ids = torch.cat([generated_ids, next_id], dim=1)

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = torch.cat([
            attention_mask,
            torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device),
        ], dim=1)
        input_encoded = {"input_ids": next_id, "attention_mask": attention_mask}

        # break if '<|endoftext|>' is generated
        if next_id.item() == tokenizer.pad_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

In [39]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_proportion(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟！我们一家人准备了20个大点的饺子，我要做热饭，20个大点的小米饺和15个大点的大饺子，那么妈妈每天做饭最少要花多少时间才能做到最少的时间？

但


In [40]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_proportion(input_text, model, tokenizer, k=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人而不能明其性，不能知其情，必不能得志。有文其心，无如是，有如是其心，其道必不，必必。这段话的意思是什么？

这是一段很难理解


Do you think the proportional sampling produces better text?

Have fun sampling! :)

---

## T3. Implement Top-p Sampling

Next, we will implement top-p sampling, which works in parallel to top-k sampling.

In `filter_topk_topp()`, we first filter out the logits that are not in the top-k, by setting their logit values to `-float('inf')`. 

And then filter out the logits whose cumulative probability (as computed from the altered logits from the previous step) is greater than `p`.
- You can first call `torch.sort()` to sort the logits in ascending order, and convert them to probabilities by applying `torch.softmax()`.
- Then, you can compute the cumulative probabilities by calling `torch.cumsum()`.
- Note that it is possible that the first logit alone dominates the distribution, and its cumulative probability is greater than `p`. In this case, we want to keep this logit, and remove all other logits.

In [41]:
def filter_topk_topp(logits: torch.Tensor, k=50, p=0.9) -> torch.Tensor: 
    '''
    Filter a distribution of logits using top-k and/or nucleus (top-p) filtering
    '''
    assert logits.dim() == 1
    logits = logits.clone()

    if k > 0:
        ### START YOUR CODE ###
        topk_logits, topk_indices = torch.topk(logits, k=min(k, logits.size(-1))) # use topk()
        logits_to_remove = torch.ones_like(logits, dtype=torch.bool) # find the indices for the logits to be removed
        logits_to_remove[topk_indices] = False
        logits[logits_to_remove] = -float('Inf')
        # you can use alternative ways
        ### END YOUR CODE ###
    
    if p > 0.0:
        ### START YOUR CODE ###
        logits_sorted, indices_sorted = torch.sort(logits, descending=True) # use torch.sort()
        cum_probs = torch.cumsum(torch.softmax(logits_sorted, dim=-1), dim=-1) # use torch.softmax() and torch.cumsum()
        indices_to_remove = indices_sorted[cum_probs > p] # find the indices for the logits to be removed

        # It is possible that cum_probs[0] > p, in which case all logits will be removed
        # we want to avoid that, so always keep the first logit
        if indices_to_remove.shape[0] == indices_sorted.shape[0]:
            indices_to_remove = indices_sorted[1:]

        logits[indices_to_remove] = -float('Inf')
        ### END YOUR CODE ### 


    return logits

In [42]:
# Test filter_topk_topp
logits = torch.tensor(list(range(10))).float()
print('original logits:', logits)

logits2 = filter_topk_topp(logits, k=5, p=0.0)
print('\nk=5, p=0.0:', logits2)

logits3 = filter_topk_topp(logits, k=0, p=0.9)
print('\nk=0, p=0.9:', logits3)

logits4 = filter_topk_topp(logits, k=0, p=0.9999999)
print('\nk=0, p=0.9999999:', logits4)

logits5 = filter_topk_topp(logits, k=5, p=0.9999999)
print('\nk=5, p=0.9999999:', logits5)


# You are expected to see the following output:
# original logits: tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])
# k=5, p=0.0: tensor([-inf, -inf, -inf, -inf, -inf, 5., 6., 7., 8., 9.])
# k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 8., 9.])
# k=0, p=0.9999999: tensor([-inf, 1., 2., 3., 4., 5., 6., 7., 8., 9.])
# k=5, p=0.9999999: tensor([-inf, -inf, -inf, -inf, -inf, 5., 6., 7., 8., 9.])

original logits: tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])

k=5, p=0.0: tensor([-inf, -inf, -inf, -inf, -inf, 5., 6., 7., 8., 9.])

k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 8., 9.])

k=0, p=0.9999999: tensor([-inf, 1., 2., 3., 4., 5., 6., 7., 8., 9.])

k=5, p=0.9999999: tensor([-inf, -inf, -inf, -inf, -inf, -inf, 6., 7., 8., 9.])


In the following test, if all logits are `-inf`, then your top-p sampling is not correctly implemented. 

You wan to keep at least one element in the logits, whose logit value dominates the distribution. 

In [44]:
import numpy as np
logits_special = torch.tensor(np.arange(10)**2).float()
print('original logits:', logits_special)

logits6 = filter_topk_topp(logits_special, k=0, p=0.9)
print('\nk=0, p=0.9:', logits6)


# You are expected to see the following output:
# original logits: tensor([ 0.,  1.,  4.,  9., 16., 25., 36., 49., 64., 81.])
# k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 81.])

original logits: tensor([ 0.,  1.,  4.,  9., 16., 25., 36., 49., 64., 81.])

k=0, p=0.9: tensor([-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, 81.])


Finally, we integrate the filtering to the generation process.

Given the output of `filter_topk_topp()`, you need to use the same sampling algorithm as implemented in `generate_topk_proportion()`.

In [47]:
def generate_topk_topp(input_text, model, tokenizer, k=50, p=0.9, max_gen_len=20):
    '''
    Generate tokens from the top-k and top-p filtered logits, and yield the sampled token id.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded.input_ids

    # initialize KV cache
    past_key_values = DynamicCache()

    # to store generated token ids
    generated_ids = input_ids

    count = 0
    while count < max_gen_len:
        output = model(
            input_ids=input_encoded["input_ids"],
            attention_mask=input_encoded["attention_mask"],
            past_key_values=past_key_values,
            use_cache=True,
        )

        # Get last token logits
        ### START YOUR CODE ###
        last_token_logits = output.logits[0, -1, :]
        ### END YOUR CODE ###

        # Get the filtered logits by calling filter_topk_topp 
        ### START YOUR CODE ###
        filtered_logits = filter_topk_topp(last_token_logits, k=k, p=p)
        ### END YOUR CODE ###


        # Sample from the remaining tokens in sorted_logits
        ### START YOUR CODE ###
        import torch
        import torch.nn.functional as F
        filtered_probs = F.softmax(filtered_logits, dim=-1) # use softmax
        try:
            next_id = torch.multinomial(filtered_probs, num_samples=1).view(1, 1) # use torch.multinomial() or your own version of sampling from filtered_probs
        except RuntimeError:
            raise

        generated_ids = torch.cat([generated_ids, next_id], dim=1) # append next_id to generated_ids

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = torch.cat([
            attention_mask,
            torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device),
        ], dim=1)
        input_encoded = {"input_ids": next_id, "attention_mask": attention_mask}

        if next_id.item() == tokenizer.sep_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

Do it yourself to test if adjusting the `k` and `p` values change the generation quality.

In [48]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=100, p=0.95, max_gen_len=50)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟，真是的！因为现在的天气真是适合做很多好玩的事情，比如放风筝、做手工、赏花等，如果你愿意的话，可以一起来做一些活动。天气真暖和，非常适合拍照，尤其是这些花和树木的景色


In [49]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=50, p=0.9, max_gen_len=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人而不能明其志，不能明其德，不能明其志，不能明其德，不亦已乎？（《论语·述而》）

“天也，人也，天人相应，人道合一，不可说而有之。”（《庄子·大宗师》）

“道不言，德不为也，非其德，不能得之。”（《道德经》）

这些思想中，哪一项是关于人的自然状态


## T4. Implement Temperature Sampling

Finally, we can adjust `filtered_logits` returned from top-k and top-p, using the temperature $\tau$, before feeding it to softmax.

In [52]:
def generate_topk_topp(input_text, model, tokenizer, k=50, p=0.9, temperature=1.0, max_gen_len=20):
    '''
    Generate tokens from the top-k and top-p filtered logits, and yield the sampled token id.
    '''
    input_encoded = tokenizer(input_text, return_tensors="pt", add_special_tokens=False)
    input_ids = input_encoded.input_ids

    # initialize KV cache
    past_key_values = DynamicCache()

    # to store generated token ids
    generated_ids = input_ids

    count = 0
    while count < max_gen_len:
        output = model(
            input_ids=input_encoded["input_ids"],
            attention_mask=input_encoded["attention_mask"],
            past_key_values=past_key_values,
            use_cache=True,
        )

        # Get last token logits
        ### START YOUR CODE ###
        last_token_logits = output.logits[0, -1, :]
        ### END YOUR CODE ###

        # Get the filtered logits by calling filter_topk_topp 
        ### START YOUR CODE ###
        filtered_logits = filter_topk_topp(last_token_logits, k=k, p=p)
        ### END YOUR CODE ###

        # Adjust the logits with temperature
        ### START YOUR CODE ###
        filtered_logits = filtered_logits / max(temperature, 1e-5)
        ### END YOUR CODE ###

        # Sample from the remaining tokens in sorted_logits
        ### START YOUR CODE ###
        import torch
        import torch.nn.functional as F
        filtered_probs = F.softmax(filtered_logits, dim=-1)
        try:
            next_id = torch.multinomial(filtered_probs, num_samples=1).view(1, 1)
        except RuntimeError:
            raise

        generated_ids = torch.cat([generated_ids, next_id], dim=1)

        # update attention mask and the `input_encoded` dictionary
        attention_mask = input_encoded["attention_mask"]
        attention_mask = torch.cat([
            attention_mask,
            torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device),
        ], dim=1)
        input_encoded = {"input_ids": next_id, "attention_mask": attention_mask}

        if next_id.item() == tokenizer.sep_token_id:
            break
        ### END YOUR CODE ###
        count += 1
    
    # reset cache
    past_key_values.reset()
    
    return generated_ids

Do it yourself to see if using higher temperature can lead to more creative generations.

In [53]:
# Test 1
torch.manual_seed(123)
input_text = "今天天气不错哟"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=100, p=0.95, temperature=0.5, max_gen_len=50)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 今天天气不错哟
generated text:
今天天气不错哟，真是的，我今天要写一篇关于天气的作文，题目是“我最喜欢的天气”，写的时候要以第一人称，描述自己最喜欢那一片天气，然后在结尾处加入一句关于天气的谚语。作文需要


In [54]:
# Test 2
torch.manual_seed(123)
input_text = "子曰：人"

generated_ids = generate_topk_topp(input_text, model, tokenizer, k=50, p=0.95, temperature=0.9, max_gen_len=100)

print("input text:", input_text)
print("generated text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

input text: 子曰：人
generated text:
子曰：人之为人都有为，为己也，不以己自任，与己为他人之道，然后可以以己为己，为己，为己，为己也。什么意思？

这句话的意思。我来解释。首先，先说“人之为人都有为”。“为”是一个动词，“有为”应该是指有作为、有成就。这句话的意思是说，每个人都有自己的作为和目标，因此，每个人都有自己的发展。

然后，“为


Congratulations! You have completed the lab for top-k and top-p sampling.